# Python Exception Handling Cheat Sheet

## 1. What is an Exception?

An **exception** is an error that occurs at runtime and stops the program.

Exception handling lets you catch those errors and respond gracefully instead of crashing.

## 2. Common Built-in Exceptions

| Exception | Cause |
|---|---|
| `ValueError` | Wrong type of value (`int("abc")`) |
| `TypeError` | Wrong type in operation (`"a" + 1`) |
| `ZeroDivisionError` | Dividing by zero |
| `IndexError` | List index out of range |
| `KeyError` | Dict key not found |
| `FileNotFoundError` | File does not exist |
| `AttributeError` | Object has no such attribute |
| `NameError` | Variable not defined |
| `ImportError` | Module not found |

## 3. Basic try / except

In [ ]:
try:
    result = 10 / 0
except ZeroDivisionError:
    print("Cannot divide by zero!")

## 4. Catching Multiple Exceptions

In [ ]:
def safe_divide(a, b):
    try:
        return a / b
    except ZeroDivisionError:
        print("Error: division by zero")
    except TypeError:
        print("Error: both arguments must be numbers")

safe_divide(10, 2)    # 5.0
safe_divide(10, 0)    # Error: division by zero
safe_divide(10, "x") # Error: both arguments must be numbers

## 5. The else Clause

`else` runs only if **no exception** was raised.

In [ ]:
try:
    num = int(input("Enter a number: "))
except ValueError:
    print("Not a valid number.")
else:
    print(f"You entered: {num}")  # only runs if no error

## 6. The finally Clause

`finally` **always runs**, whether or not an exception occurred. Used for cleanup (closing files, DB connections).

In [ ]:
try:
    f = open("data.txt", "r")
    content = f.read()
except FileNotFoundError:
    print("File not found.")
finally:
    print("Cleanup done.")  # always runs

## 7. Full Structure: try / except / else / finally

In [ ]:
try:
    num = int(input("Enter an integer: "))
except ValueError as e:
    print(f"Caught: {e}")
else:
    print(f"Squared: {num ** 2}")
finally:
    print("--- done ---")

## 8. Accessing the Exception Object

Use `as e` to capture the exception and inspect its message.

In [ ]:
try:
    my_list = [1, 2, 3]
    print(my_list[10])
except IndexError as e:
    print(f"IndexError: {e}")
    print(f"Type: {type(e).__name__}")

## 9. Raising Exceptions

Use `raise` to throw an exception deliberately.

In [ ]:
def set_age(age):
    if age < 0:
        raise ValueError(f"Age cannot be negative: {age}")
    return age

try:
    set_age(-5)
except ValueError as e:
    print(e)

## 10. Custom Exceptions

Create your own exception classes by inheriting from `Exception`.

In [ ]:
class InsufficientFundsError(Exception):
    def __init__(self, balance, amount):
        super().__init__(f"Cannot withdraw {amount}. Balance is only {balance}.")
        self.balance = balance
        self.amount = amount

def withdraw(balance, amount):
    if amount > balance:
        raise InsufficientFundsError(balance, amount)
    return balance - amount

try:
    withdraw(100, 250)
except InsufficientFundsError as e:
    print(e)

## 11. Catching All Exceptions (Use Sparingly)

Catching `Exception` is a catch-all. Avoid hiding real bugs.

In [ ]:
try:
    risky_code = 1 / 0
except Exception as e:
    print(f"Something went wrong: {type(e).__name__}: {e}")

---
## Practice Challenges

### Challenge 1: Safe Integer Converter
Write a function that converts a string to int and returns `None` on failure.

In [ ]:
def safe_int(value):
    try:
        return int(value)
    except (ValueError, TypeError):
        return None

print(safe_int("42"))    # 42
print(safe_int("abc"))   # None
print(safe_int(None))    # None

### Challenge 2: File Reader
Read a file safely. Print its contents or a friendly error if it doesn't exist.

In [ ]:
def read_file(path):
    try:
        with open(path, "r") as f:
            return f.read()
    except FileNotFoundError:
        return f"File not found: {path}"
    except PermissionError:
        return f"Permission denied: {path}"

print(read_file("notes.txt"))
print(read_file("nonexistent.txt"))

### Challenge 3: Validated User Age
Ask for age with `input()`. Validate it is an integer between 0 and 120, re-prompting on any error.

In [ ]:
while True:
    try:
        age = int(input("Enter age: "))
        if not (0 <= age <= 120):
            raise ValueError("Age must be between 0 and 120.")
        break
    except ValueError as e:
        print(f"Invalid: {e}")

print(f"Age accepted: {age}")

### Challenge 4: Custom BankAccount with Exceptions
Implement a `BankAccount` that raises custom exceptions for overdraft and negative deposit.

In [ ]:
class NegativeAmountError(Exception): pass
class OverdraftError(Exception): pass

class BankAccount:
    def __init__(self, balance=0):
        self.balance = balance

    def deposit(self, amount):
        if amount <= 0:
            raise NegativeAmountError("Deposit must be positive.")
        self.balance += amount

    def withdraw(self, amount):
        if amount <= 0:
            raise NegativeAmountError("Withdrawal must be positive.")
        if amount > self.balance:
            raise OverdraftError(f"Insufficient funds. Balance: {self.balance}")
        self.balance -= amount

account = BankAccount(200)
try:
    account.deposit(-50)
except NegativeAmountError as e:
    print(e)

try:
    account.withdraw(500)
except OverdraftError as e:
    print(e)

account.deposit(100)
account.withdraw(50)
print(f"Final balance: {account.balance}")

### Challenge 5: Exception Chaining with raise...from
Re-raise a low-level exception as a higher-level one with context preserved.

In [ ]:
class ConfigError(Exception): pass

def load_config(path):
    try:
        with open(path) as f:
            return f.read()
    except FileNotFoundError as e:
        raise ConfigError(f"Config file missing: {path}") from e

try:
    load_config("config.yaml")
except ConfigError as e:
    print(f"Config error: {e}")
    print(f"Original cause: {e.__cause__}")